In [1]:
def get_numpy_data(dataframe, features, output):
    df_temp = dataframe.copy()
    df_temp['constant'] = 1.0
    all_features = ['constant'] + features
    feature_matrix = df_temp[all_features].to_numpy()
    output_array = df_temp[output].to_numpy()
    return feature_matrix, output_array

import pandas as pd
import numpy as np
sales = pd.read_csv('kc_house_data.csv')

In [2]:
def normalize_features(feature_matrix):
    norms = np.linalg.norm(feature_matrix, axis=0)
    
    normalized_features = feature_matrix / norms
    return normalized_features, norms

test_matrix = np.array([[3, 6, 9], [4, 8, 12]])
norm_feat, norms = normalize_features(test_matrix)

print("Перевірка Завдання 1:")
print(f"Норми: {norms}") 
print(f"Нормалізована матриця:\n{norm_feat}") 

Перевірка Завдання 1:
Норми: [ 5. 10. 15.]
Нормалізована матриця:
[[0.6 0.6 0.6]
 [0.8 0.8 0.8]]


In [3]:
def lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty):
    prediction = np.dot(feature_matrix, weights)
    
    feature_i = feature_matrix[:, i]
    rho_i = np.dot(feature_i, (output - prediction + weights[i] * feature_i))

    if i == 0:
        new_weight = rho_i
    elif rho_i < -l1_penalty/2:
        new_weight = rho_i + l1_penalty/2
    elif rho_i > l1_penalty/2:
        new_weight = rho_i - l1_penalty/2
    else:
        new_weight = 0.
        
    return new_weight

from math import sqrt

test_feat = np.array([[3/sqrt(13), 1/sqrt(10)], [2/sqrt(13), 3/sqrt(10)]])
test_output = np.array([1, 1])
test_weights = np.array([1., 4.])

res = lasso_coordinate_descent_step(i=1, feature_matrix=test_feat, 
                                    output=test_output, weights=test_weights, 
                                    l1_penalty=0.1)

print(f"Результат кроку (очікується ≈0.4256): {res:.4f}")

Результат кроку (очікується ≈0.4256): 0.4256


In [4]:
def lasso_cyclical_coordinate_descent(feature_matrix, output, initial_weights, l1_penalty, tolerance):
    weights = np.array(initial_weights)
    converged = False
    
    while not converged:
        max_change = 0
      
        for i in range(len(weights)):
            old_weight = weights[i]
            
            weights[i] = lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty)
            
            change = abs(weights[i] - old_weight)
            if change > max_change:
                max_change = change
                
        if max_change < tolerance:
            converged = True
            
    return weights

In [5]:
features = ['sqft_living', 'bedrooms']
my_output = 'price'

(feature_matrix_raw, output_array) = get_numpy_data(sales, features, my_output)

normalized_features, norms = normalize_features(feature_matrix_raw)

initial_weights = np.array([0., 0., 0.])
tolerance = 1.0

print("Запуск розрахунку для L1_penalty = 1e7...")
weights_1e7 = lasso_cyclical_coordinate_descent(normalized_features, output_array, initial_weights, 1e7, tolerance)
prediction_1e7 = np.dot(normalized_features, weights_1e7)
rss_1e7 = np.sum((prediction_1e7 - output_array)**2)

print("Запуск розрахунку для L1_penalty = 1e8...")
weights_1e8 = lasso_cyclical_coordinate_descent(normalized_features, output_array, initial_weights, 1e8, tolerance)
prediction_1e8 = np.dot(normalized_features, weights_1e8)
rss_1e8 = np.sum((prediction_1e8 - output_array)**2)

def print_lasso_results(penalty, weights, rss, features_list):
    print(f"\n" + "="*40)
    print(f"РЕЗУЛЬТАТИ ДЛЯ L1_PENALTY = {penalty:.0e}")
    print(f"="*40)
    print(f"Навчені ваги: {weights}")
    print(f"RSS на нормалізованому наборі: {rss:.2e}")
    
    all_feat_names = ['constant'] + features_list
    active_features = [all_feat_names[i] for i, w in enumerate(weights) if w != 0]
    print(f"Ознаки з ненульовими вагами: {active_features}")

print_lasso_results(1e7, weights_1e7, rss_1e7, features)
print_lasso_results(1e8, weights_1e8, rss_1e8, features)

Запуск розрахунку для L1_penalty = 1e7...
Запуск розрахунку для L1_penalty = 1e8...

РЕЗУЛЬТАТИ ДЛЯ L1_PENALTY = 1e+07
Навчені ваги: [21624997.9595191  63157247.20788956        0.        ]
RSS на нормалізованому наборі: 1.63e+15
Ознаки з ненульовими вагами: ['constant', 'sqft_living']

РЕЗУЛЬТАТИ ДЛЯ L1_PENALTY = 1e+08
Навчені ваги: [79400304.6376446        0.               0.       ]
RSS на нормалізованому наборі: 2.91e+15
Ознаки з ненульовими вагами: ['constant']
